# Leitura dos dados no S3 — Bronze, Silver e Gold

Lê cada dataset das três camadas com pandas, usando **awswrangler** sobre o cliente S3 do
boto3. O awswrangler resolve o prefixo, reconstrói as colunas de partição (`ano`) e devolve
um DataFrame do pandas.

**Instalação**

```bash
pip install awswrangler pandas pyarrow
```

**Credenciais** — perfil `academy` em `~/.aws/credentials` ou `~/.aws/config`, com token temporário.
As credenciais do AWS Academy expiram junto com a sessão do lab; quando isso acontecer,
copie as novas e reexecute a célula de configuração. O arquivo tem esta forma:

```ini
[academy]
aws_access_key_id = ...
aws_secret_access_key = ...
aws_session_token = ...
```

As três linhas são obrigatórias — copiar só `key` e `secret`, sem o `session_token`,
falha com `InvalidClientTokenId`.

**Volume** — `alunos` tem ~3,9M linhas. Nessas bases o notebook usa `amostra()`, que lê só
o primeiro bloco em vez de baixar o dataset inteiro.

**Ordem** — silver e gold só existem depois dos jobs correspondentes rodarem.

## Configuração

In [14]:
import boto3
import awswrangler as wr
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

# Perfil com token temporario do AWS Academy, definido em ~/.aws/credentials.
# A sessao e passada explicitamente para o awswrangler nos helpers abaixo:
# sem isso ele usaria a sessao default, que nao carrega este perfil.
PERFIL = "academy"

SESSAO = boto3.Session(profile_name=PERFIL)

BRONZE = "s3://tc2-bronze-1"
SILVER = "s3://tc2-silver-1"
GOLD = "s3://tc2-gold-1"

print("awswrangler", wr.__version__)
print("perfil      ", PERFIL)
print("regiao      ", SESSAO.region_name)

awswrangler 3.14.0
perfil       academy
regiao       None


In [ ]:
# Confirma que o token ainda vale antes de tentar ler qualquer bucket.
try:
    identidade = SESSAO.client("sts").get_caller_identity()
    print("conta     :", identidade["Account"])
    print("identidade:", identidade["Arn"])
except Exception as erro:
    print("Falha ao autenticar:", type(erro).__name__, erro)
    print()
    print("O token do AWS Academy expira com a sessao do lab.")
    print(f"Copie as credenciais novas para ~/.aws/credentials no perfil [{PERFIL}]")
    print("e reexecute esta celula.")

In [ ]:
def ler(caminho, colunas=None, filtro_particao=None):
    """Lê o dataset inteiro do prefixo.

    `dataset=True` faz o awswrangler tratar o prefixo como um conjunto de
    arquivos e reconstruir as colunas de partição a partir do caminho.

    `filtro_particao` recebe um lambda sobre o dict de partições, para ler
    só parte do dataset:
    `filtro_particao=lambda p: p["ano"] == "2024"`.

    Atenção: os valores das partições chegam como **string**.
    """
    return wr.s3.read_parquet(
        path=caminho,
        dataset=True,
        columns=colunas,
        partition_filter=filtro_particao,
        boto3_session=SESSAO,
    )


def amostra(caminho, n=5000, colunas=None):
    """Lê apenas o primeiro bloco de n linhas, sem baixar o dataset todo.

    Atenção: é o *primeiro* bloco, não uma amostra do conjunto. Em dataset
    particionado ele costuma cair numa única partição, então não use esta
    função para contar ou agrupar por coluna de partição -- os outros valores
    simplesmente não aparecem. Para isso use `ler()`.
    """
    blocos = wr.s3.read_parquet(
        path=caminho,
        dataset=True,
        columns=colunas,
        chunked=n,
        boto3_session=SESSAO,
    )
    return next(blocos)


def listar(caminho):
    """Objetos sob o prefixo. Útil para conferir se o job já gravou."""
    return wr.s3.list_objects(caminho, boto3_session=SESSAO)


def resumo(df, nome):
    """Imprime dimensões e tipos, devolve o head para o notebook exibir."""
    print(f"{nome}: {len(df):,} linhas x {df.shape[1]} colunas\n")
    print(df.dtypes.to_string())
    return df.head()

### O que existe em cada bucket

Rode antes das seções seguintes para ver quais camadas já foram materializadas.

In [ ]:
for camada in (BRONZE, SILVER, GOLD):
    pastas = wr.s3.list_directories(f"{camada}/", boto3_session=SESSAO)
    print(f"\n{camada}")
    for pasta in sorted(pastas):
        print("   ", pasta.rstrip("/").rsplit("/", 1)[-1])

# 1. Bronze

Cópia fiel da origem, sem transformação. Gravada pelas lambdas de ingestão em
`s3://tc2-bronze/{tabela}/{tabela}.parquet`. Não é particionada.

## 1.1 municipiosIBGE

Municípios do IBGE, fonte de `nome_municipio` e `sigla_uf` para as silvers de município e alunos. Ingerida pela lambda `cargaTabelaMunicipio`.

In [ ]:
bronze_municipios_ibge = ler(f"{BRONZE}/municipiosIBGE/")
resumo(bronze_municipios_ibge, "bronze/municipiosIBGE")

## 1.2 dicionario

De-para dos códigos de todas as tabelas: `id_tabela`, `nome_coluna`, `chave`, `valor`. É a fonte das colunas `*_descricao` nas silvers.

In [ ]:
bronze_dicionario = ler(f"{BRONZE}/dicionario/")
resumo(bronze_dicionario, "bronze/dicionario")

## 1.3 uf

Taxa de alfabetização por UF, ano, série e rede.

In [ ]:
bronze_uf = ler(f"{BRONZE}/uf/")
resumo(bronze_uf, "bronze/uf")

## 1.4 municipio

Taxa de alfabetização por município, ano, série e rede.

In [ ]:
bronze_municipio = ler(f"{BRONZE}/municipio/")
resumo(bronze_municipio, "bronze/municipio")

## 1.5 meta_alfabetizacao_uf

Metas de 2024 a 2030 por UF e ano, rede Pública. Uma linha por `(sigla_uf, ano)`.

In [ ]:
bronze_meta_uf = ler(f"{BRONZE}/meta_alfabetizacao_uf/")
resumo(bronze_meta_uf, "bronze/meta_alfabetizacao_uf")

## 1.6 meta_alfabetizacao_municipio

Metas de 2024 a 2030 por município e ano, rede Municipal. Uma linha por `(id_municipio, ano)`.

In [ ]:
bronze_meta_municipio = ler(f"{BRONZE}/meta_alfabetizacao_municipio/")
resumo(bronze_meta_municipio, "bronze/meta_alfabetizacao_municipio")

## 1.7 meta_alfabetizacao_brasil

Metas nacionais de 2024 a 2030, rede Pública. Uma linha por ano.

In [ ]:
bronze_meta_brasil = ler(f"{BRONZE}/meta_alfabetizacao_brasil/")
resumo(bronze_meta_brasil, "bronze/meta_alfabetizacao_brasil")

## 1.8 alunos

Microdado no grão de aluno, ~3,9M linhas. Traz o alvo `alfabetizado` e as colunas
codificadas `presenca`, `preenchimento_caderno` e `rede`.

Lido por amostra. Para um ano inteiro:
`ler(f"{BRONZE}/alunos/", filtro_particao=lambda p: p["ano"] == "2024")`.

In [ ]:
bronze_alunos = amostra(f"{BRONZE}/alunos/", n=5000)
resumo(bronze_alunos, "bronze/alunos (amostra)")

# 2. Silver

Dados limpos e enriquecidos: tipos garantidos, códigos traduzidos pelo dicionário,
município e UF vindos do IBGE, e metas de alfabetização anexadas.

Particionada por `ano`. O valor também fica dentro do arquivo, em `ano_referencia`,
para o parquet ser autocontido se lido arquivo a arquivo.

## 2.1 uf

Uma linha por `(sigla_uf, ano, serie, rede)`. Metas em `meta_alfabetizacao_uf_2024..2030`
e `meta_alfabetizacao_brasil_2024..2030`, preenchidas **apenas na rede 5** (Pública) —
nas outras redes ficam nulas, para não comparar a taxa de uma rede com a meta de outra.

In [ ]:
silver_uf = ler(f"{SILVER}/uf/")
resumo(silver_uf, "silver/uf")

## 2.2 municipio

Uma linha por `(id_municipio, ano, serie, rede)`, com `nome_municipio` e `sigla_uf`.
`meta_alfabetizacao_municipio_*` só na rede 3 (Municipal) e `meta_alfabetizacao_brasil_*`
só na rede 5 (Pública).

In [ ]:
silver_municipio = ler(f"{SILVER}/municipio/")
resumo(silver_municipio, "silver/municipio")

## 2.3 alunos

Microdado enriquecido: município e UF do IBGE, e as cinco colunas `*_descricao`
(`rede`, `serie`, `presenca`, `preenchimento_caderno`, `alfabetizado`). Sem metas —
não existe base de meta no grão de aluno.

In [ ]:
silver_alunos = amostra(f"{SILVER}/alunos/", n=5000)
resumo(silver_alunos, "silver/alunos (amostra)")

# 3. Gold

Datasets analíticos gravados por `glue/silver_to_gold/gold_job.py`.

Cada nível geográfico tem sua própria pasta e mantém as colunas nativas do nível:
`indicador_municipio` tem `id_municipio`, `indicador_uf` tem `sigla_uf`, e
`indicador_brasil` não tem chave geográfica. Todos particionados por `ano`.

## 3.1 indicador_municipio

Uma linha por `(id_municipio, ano, serie, rede)`, com `taxa_alfabetizacao`,
`media_portugues`, `qtd_alunos` e as metas municipal e nacional.
`origem_indicador = 'inep_publicado'`.

Traz as taxas de referência desnormalizadas na própria linha —
`taxa_alfabetizacao_uf`, `taxa_alfabetizacao_brasil`, `diferenca_pp_vs_uf` e
`diferenca_pp_vs_brasil` — casadas por ano, série e rede. Assim o dashboard responde
"esse município está acima ou abaixo do seu estado e do país" sem nenhum join.

In [15]:
gold_indicador_municipio = ler(f"{GOLD}/indicador_municipio/")
resumo(gold_indicador_municipio, "gold/indicador_municipio")

gold/indicador_municipio: 23,995 linhas x 21 colunas

id_municipio                 string[python]
nome_municipio               string[python]
sigla_uf                     string[python]
ano_referencia                        Int32
serie                                 Int32
serie_descricao              string[python]
rede                                  Int32
rede_descricao               string[python]
taxa_alfabetizacao                  float64
media_portugues                     float64
origem_indicador             string[python]
qtd_alunos                            Int64
meta_alfabetizacao_do_ano           float64
gap_meta_do_ano_pp                  float64
taxa_alfabetizacao_uf               float64
diferenca_pp_vs_uf                  float64
taxa_alfabetizacao_brasil           float64
diferenca_pp_vs_brasil              float64
nivel_geografico             string[python]
processed_at                 datetime64[ns]
ano                                category


,id_municipio,nome_municipio,sigla_uf,ano_referencia,serie,serie_descricao,rede,rede_descricao,taxa_alfabetizacao,media_portugues,origem_indicador,qtd_alunos,meta_alfabetizacao_do_ano,gap_meta_do_ano_pp,taxa_alfabetizacao_uf,diferenca_pp_vs_uf,taxa_alfabetizacao_brasil,diferenca_pp_vs_brasil,nivel_geografico,processed_at,ano
0,2311801,Russas,CE,2023,2,2° ano do Ensino Fundamental,3,Municipal,92.93,804.7985,inep_publicado,995,NaN,NaN,84.49,8.44,57.205024,35.724976,municipio,2026-08-28 22:57:35.870596,2023
1,2910602,Esplanada,BA,2023,2,2° ano do Ensino Fundamental,5,Pública (Estadual e Municipal),19.87,703.8273,inep_publicado,363,NaN,NaN,36.78,-16.91,57.462163,-37.592163,municipio,2026-08-28 22:57:35.870596,2023
2,2913507,Iguaí,BA,2023,2,2° ano do Ensino Fundamental,3,Municipal,49.55,739.9796,inep_publicado,186,NaN,NaN,36.77,12.78,57.205024,-7.655024,municipio,2026-08-28 22:57:35.870596,2023
3,4101408,Apucarana,PR,2023,2,2° ano do Ensino Fundamental,5,Pública (Estadual e Municipal),82.14,767.9827,inep_publicado,1464,NaN,NaN,73.12,9.02,57.462163,24.677837,municipio,2026-08-28 22:57:35.870596,2023
4,4201901,Aurora,SC,2023,2,2° ano do Ensino Fundamental,3,Municipal,62.58,759.2888,inep_publicado,33,NaN,NaN,62.04,0.54,57.205024,5.374976,municipio,2026-08-28 22:57:35.870596,2023


## 3.2 indicador_uf

Mesma estrutura no grão de UF, com as metas de UF e do Brasil, `qtd_alunos` e o
comparativo contra o país (`taxa_alfabetizacao_brasil`, `diferenca_pp_vs_brasil`).

`qtd_alunos` vem do microdado de alunos, agregado por UF, ano, série e rede. Como o
microdado não traz a rede 1 (Federal), a contagem da rede 0 (Total) sai sem essa parcela.

In [16]:
gold_indicador_uf = ler(f"{GOLD}/indicador_uf/")
resumo(gold_indicador_uf, "gold/indicador_uf")

gold/indicador_uf: 145 linhas x 17 colunas

sigla_uf                     string[python]
ano_referencia                        Int32
serie                                 Int32
serie_descricao              string[python]
rede                                  Int32
rede_descricao               string[python]
taxa_alfabetizacao                  float64
media_portugues                     float64
origem_indicador             string[python]
qtd_alunos                            Int64
meta_alfabetizacao_do_ano           float64
gap_meta_do_ano_pp                  float64
taxa_alfabetizacao_brasil           float64
diferenca_pp_vs_brasil              float64
nivel_geografico             string[python]
processed_at                 datetime64[ns]
ano                                category


,sigla_uf,ano_referencia,serie,serie_descricao,rede,rede_descricao,taxa_alfabetizacao,media_portugues,origem_indicador,qtd_alunos,meta_alfabetizacao_do_ano,gap_meta_do_ano_pp,taxa_alfabetizacao_brasil,diferenca_pp_vs_brasil,nivel_geografico,processed_at,ano
0,PE,2023,2,2° ano do Ensino Fundamental,5,Pública (Estadual e Municipal),58.95,747.4522,inep_publicado,88271,NaN,NaN,57.462163,1.487837,uf,2026-08-28 22:57:34.773484,2023
1,AL,2023,2,2° ano do Ensino Fundamental,5,Pública (Estadual e Municipal),43.88,729.7227,inep_publicado,35503,NaN,NaN,57.462163,-13.582163,uf,2026-08-28 22:57:34.773484,2023
2,MT,2023,2,2° ano do Ensino Fundamental,5,Pública (Estadual e Municipal),55.03,746.8000,inep_publicado,54526,NaN,NaN,57.462163,-2.432163,uf,2026-08-28 22:57:34.773484,2023
3,MS,2023,2,2° ano do Ensino Fundamental,2,Estadual,42.90,729.3908,inep_publicado,768,NaN,NaN,60.093454,-17.193454,uf,2026-08-28 22:57:34.773484,2023
4,MA,2023,2,2° ano do Ensino Fundamental,2,Estadual,72.38,768.0339,inep_publicado,175,NaN,NaN,60.093454,12.286546,uf,2026-08-28 22:57:34.773484,2023


## 3.3 indicador_brasil

Uma linha por `(ano, serie, rede)`, com `qtd_alunos`. A taxa aqui é **calculada** da silver
de alunos — média de `alfabetizado` ponderada por `peso_aluno` — e por isso sai com
`origem_indicador = 'calculado_microdado'`. Precisa ser reconciliada com a taxa nacional
publicada pelo INEP antes de virar indicador oficial.

In [ ]:
gold_indicador_brasil = ler(f"{GOLD}/indicador_brasil/")
resumo(gold_indicador_brasil, "gold/indicador_brasil")

## 3.4 metas_vs_resultado

Formato longo: uma linha por geografia, ano observado, escopo da meta e ano da meta,
com `gap_pp`, `percentual_da_meta`, `atingiu_meta` e `meta_ja_vencida`.

`escopo_meta` diz de qual meta a linha fala: no nível de município vem `municipio` e
`brasil`; no de UF, `uf` e `brasil`; no de Brasil, só `brasil`.

In [ ]:
gold_metas_municipio = ler(f"{GOLD}/metas_vs_resultado_municipio/")
resumo(gold_metas_municipio, "gold/metas_vs_resultado_municipio")

In [ ]:
gold_metas_uf = ler(f"{GOLD}/metas_vs_resultado_uf/")
resumo(gold_metas_uf, "gold/metas_vs_resultado_uf")

In [ ]:
gold_metas_brasil = ler(f"{GOLD}/metas_vs_resultado_brasil/")
resumo(gold_metas_brasil, "gold/metas_vs_resultado_brasil")

In [ ]:
# linhas por escopo de meta em cada nível
for nome, df in [("municipio", gold_metas_municipio),
                 ("uf", gold_metas_uf),
                 ("brasil", gold_metas_brasil)]:
    print(nome, dict(df["escopo_meta"].value_counts()))

## 3.5 evolucao_temporal

Série por geografia, rede e série escolar, com `taxa_ano_anterior`, `variacao_pp`,
`variacao_percentual`, `variacao_acumulada_pp` e `gap_para_meta_2030`.

In [ ]:
gold_evolucao_municipio = ler(f"{GOLD}/evolucao_temporal_municipio/")
resumo(gold_evolucao_municipio, "gold/evolucao_temporal_municipio")

In [ ]:
gold_evolucao_uf = ler(f"{GOLD}/evolucao_temporal_uf/")
resumo(gold_evolucao_uf, "gold/evolucao_temporal_uf")

In [ ]:
gold_evolucao_brasil = ler(f"{GOLD}/evolucao_temporal_brasil/")
resumo(gold_evolucao_brasil, "gold/evolucao_temporal_brasil")

## 3.6 ml_aluno

Feature table no grão de aluno para classificação: target `alfabetizado` e
`dataset_split` (`treino` / `validacao` / `teste`), estável entre execuções porque vem do
hash do `id_aluno`.

O contexto do município é do **ano anterior**, para não vazar o alvo — a taxa do próprio
ano é calculada a partir dos mesmos alunos que formam o target.

`id_aluno`, `id_escola` e `id_municipio` são chaves para juntar a predição de volta,
**não features**: não inclua na matriz de treino.

In [ ]:
gold_ml_aluno = amostra(f"{GOLD}/ml_aluno/", n=5000)
resumo(gold_ml_aluno, "gold/ml_aluno (amostra)")

In [ ]:
# proporção dos splits (a amostra não é representativa do total)
gold_ml_aluno["dataset_split"].value_counts(normalize=True)

In [ ]:
# para treinar, leia um split por vez em vez do dataset inteiro
# treino = ler(
#     f"{GOLD}/ml_aluno/",
#     filtro_particao=lambda p: p["ano"] == "2024",
# )
# treino = treino[treino["dataset_split"] == "treino"]